# CLIP-Guided StyleGAN2 with 3 Learned Mapping Networks

## Overview

This notebook implements a pipeline where **3 separate mapping networks** (initialised from the pretrained StyleGAN2 mapping network) learn to convert CLIP image embeddings into the W+ latent space of a **StyleGAN2 synthesis network** to generate **256×256** face images.

### Architecture

```
Real Image ──► CLIP ViT-B/32 ──► 512-dim embedding ──► 3 Mapping Networks ──► delta_w (Mt(w)) ──► w_avg + Mt(w) ──► Synthesis Network ──► Generated Image
                                                                                                                                              │
                                  ┌───────────────────────────────────────────────────────────────────────────────────────────────────────────┘
                                  │
                                  ├──► Discriminator ──────────────────────────────────────────────────────► Adversarial Loss  (L_adv)
                                  │
                                  └──► CLIP ──► Cosine Distance vs Real Image Embedding ──────────────────► CLIP Loss          (L_CLIP)
                                                                                                                                │
                               Mt(w) ────────────────────────────────────────────────────────────────────► L2 Regularisation   (L_L2)
```

### Mapper-to-Layer Assignments

For 256×256 resolution, StyleGAN2 uses **14 style layers** (num_ws=14). Each mapper produces a **delta w** vector that is broadcast to its assigned layers:

| Mapper | Resolution Blocks | W Layers | # of w values |
|--------|------------------|----------|---------------|
| Mapper 0 | 4×4, 8×8 | 0, 1, 2, 3 | 4 × 512 |
| Mapper 1 | 16×16, 32×32 | 4, 5, 6, 7 | 4 × 512 |
| Mapper 2 | 64×64, 128×128, 256×256 | 8–13 | 6 × 512 |

**Total: 14 × 512 = 7,168 w values** forming the complete W+ delta tensor.

### Loss Function (StyleCLIP-style)

$$\mathcal{L} = \mathcal{L}_{adv} + \mathcal{L}_{CLIP} + \lambda_{L2} \cdot \mathcal{L}_{L2}$$

- $\mathcal{L}_{adv}$: Non-saturating generator loss from frozen Discriminator
- $\mathcal{L}_{CLIP}(w) = D_{CLIP}(G(w_{avg} + M_t(w)),\; x_{real}) = 1 - \cos(\text{CLIP}(G(w_{avg}+M_t(w))),\; \text{CLIP}(x_{real}))$
- $\mathcal{L}_{L2} = \|M_t(w)\|_2^2$ — penalises large steps in latent space
- $\lambda_{L2}$: Weight for the L2 regularisation term (default 0.8)

### What is Frozen vs Trainable

| Component | Frozen? | Trainable? |
|-----------|---------|------------|
| CLIP ViT-B/32 | ✓ | ✗ |
| StyleGAN2 Synthesis Network | ✓ | ✗ |
| StyleGAN2 Discriminator | ✓ | ✗ |
| 3 Mapping Networks | ✗ | ✓ |


In [1]:
!pip install torch torchvision ftfy regex tqdm
!pip install ninja matplotlib pillow
!pip install wandb
!pip install tqdm

## Step 1 – Install Dependencies & Clone StyleGAN2 Repository

Install PyTorch, torchvision, CLIP dependencies, and clone the StyleGAN2-ADA-PyTorch repository. We use a fork with updated dependencies for compatibility with recent PyTorch versions. The CLIP model (ViT-B/32) is also installed from OpenAI's repository.

In [2]:
!rm -rf stylegan2-ada-pytorch
!git clone https://github.com/RashmikaDushan/stylegan2-ada-pytorch.git

Cloning into 'stylegan2-ada-pytorch'...
remote: Enumerating objects: 256, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 256 (delta 57), reused 60 (delta 30), pack-reused 158 (from 3)
Receiving objects: 100% (256/256), 1.40 MiB | 1.62 MiB/s, done.
Resolving deltas: 100% (124/124), done.


In [3]:
%cd stylegan2-ada-pytorch
!git checkout dependency-updates

/home/m2s/mind2sketch/Mapping-Neural-Networks/stylegan2-ada-pytorch
branch 'dependency-updates' set up to track 'origin/dependency-updates'.
Switched to a new branch 'dependency-updates'


In [4]:
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-wroboqc9
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-wroboqc9
  Resolved https://github.com/openai/CLIP.git to commit ded190a052fdf4585bd685cee5bc96e0310d2c93
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
!pip install click

## Step 2 – Import Libraries

Config to access the pretrained StyleGAN2 checkpoint (`.pkl` file) and the image dataset.. Then import all required Python libraries including PyTorch, CLIP, and the StyleGAN2-ADA modules (`dnnlib`, `legacy`).

In [6]:
import os
import sys
import copy
import math
import torch
import clip
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms, utils
from PIL import Image

In [7]:
sys.path.append('./stylegan2-ada-pytorch')

import dnnlib
import legacy

In [8]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


## Step 3 – Load CLIP Model (ViT-B/32) and Set Device

Set the compute device to GPU (`cuda`) if available, otherwise CPU.

Load the **CLIP ViT-B/32** model which produces **512-dimensional** image embeddings. The CLIP model is **frozen** — its parameters are not updated during training. It serves two purposes:
1. **Encode real images** from the dataset into 512-dim embeddings (input to the mapping networks)
2. **Encode generated images** to compute the CLIP similarity loss against real image embeddings

In [9]:
clip_model, _ = clip.load("ViT-B/32", device=device)
clip_model.eval()

for p in clip_model.parameters():
    p.requires_grad = False

## Step 4 – Configuration & Hyperparameters

Set the paths and training hyperparameters:
- **`CHECKPOINT_PATH`**: Path to the pretrained StyleGAN2 `.pkl` file on Google Drive (trained on FFHQ at 256×256)
- **`FFHQ_DIR`**: Path to the face image dataset used as real images
- **`BATCH_SIZE`**: Number of images per training step
- **`LR`**: Learning rate for Adam optimizer (only applied to the 3 mapping networks)
- **`LAMBDA_CLIP`**: Weight for the CLIP similarity loss term in the total loss: $\mathcal{L} = \mathcal{L}_{adv} + \lambda \cdot \mathcal{L}_{CLIP}$
- **`SAVE_DIR`**: Directory to save mapper checkpoints and sample images

In [10]:
CHECKPOINT_PATH = '/home/m2s/mind2sketch/Mapping-Neural-Networks/weights/pretrained/stylegan2-ffhq-256x256.pkl'
# CHECKPOINT_PATH = '/home/m2s/mind2sketch/Mapping-Neural-Networks/weights/checkpoints/step-218750_lamda-1.4_lr-0.002_batchz-16.pt'
FFHQ_DIR = '/home/m2s/mind2sketch/Mapping-Neural-Networks/ffhq/ffhq256'
BATCH_SIZE = 32
LR = 2e-3
EPOCHS = 100
LAMBDA_L2 = 0.8        # weight for L2 regularisation on the latent manipulation step Mt(w)
VAL_SPLIT = 0.1
TEST_SPLIT = 0.1
SAVE_DIR = '/home/m2s/mind2sketch/Mapping-Neural-Networks/weights/split/3mapper_b_32'

os.makedirs(FFHQ_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)


## Step 5 – Load Pretrained StyleGAN2 (Generator & Discriminator)

Load the pretrained StyleGAN2 model from the `.pkl` checkpoint on Google Drive. This gives us:
- **`G` (Generator – G_ema)**: Contains both the **mapping network** and the **synthesis network**. We will deep-copy the mapping network 3 times to create our 3 mappers, and use the synthesis network (frozen) to generate images.
- **`D` (Discriminator)**: Used to compute the adversarial loss. Also **frozen** during training.

Both G and D are set to `eval()` mode and all their parameters have `requires_grad = False` — they are not updated during training. Only the 3 mapping networks we create will be trainable.

In [11]:
print('Loading StyleGAN2 checkpoint (must be a .pkl from stylegan2-ada)...')

if not os.path.exists(CHECKPOINT_PATH):
    print(f'WARNING: checkpoint not found at {CHECKPOINT_PATH}. Upload the checkpoint to Colab or change the path.')
    G = None
    D = None
else:
    with open(CHECKPOINT_PATH, 'rb') as f:
        print('Opening checkpoint...')
        model_data = legacy.load_network_pkl(f)

    G = model_data['G_ema'].to(device)
    D = model_data['D'].to(device)

    print('Loaded G and D from checkpoint.')

Loading StyleGAN2 checkpoint (must be a .pkl from stylegan2-ada)...
Opening checkpoint...
Loaded G and D from checkpoint.


In [12]:
if G is not None:
    for p in G.parameters():
        p.requires_grad = False
    G.eval()

if D is not None:
    for p in D.parameters():
        p.requires_grad = False
    D.eval()

## Step 6 – CLIP Encoding Function

Define a function to encode image tensors through the frozen CLIP model:

1. Convert images from `[-1, 1]` range (StyleGAN2 output format) to `[0, 1]`
2. Resize to 224×224 (CLIP's expected input size)
3. Apply CLIP's ImageNet normalization
4. Pass through `clip_model.encode_image()` to get 512-dim embeddings
5. **Cast from float16 to float32** — CLIP on CUDA outputs half-precision, but our mapping networks operate in float32. This cast fixes the `RuntimeError: mat1 and mat2 must have the same dtype` error.
6. L2-normalize the embeddings

When `allow_grad=True`, gradients flow through the CLIP encoder (needed for the generated-image path so that the CLIP loss backpropagates to the mapping networks).

In [13]:
clip_input_size = 224

def encode_with_clip_from_tensor(img_tensor, allow_grad=False):
    """
    img_tensor: (B, 3, H, W) in [-1, 1]
    Returns: (B, 512) float32 CLIP embedding, L2-normalized
    """
    x = (img_tensor + 1.0) / 2.0  # to [0, 1]
    x = F.interpolate(x, size=(clip_input_size, clip_input_size), mode='bilinear', align_corners=False)

    mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=x.device).view(1, 3, 1, 1)
    std  = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=x.device).view(1, 3, 1, 1)
    x = (x - mean) / std

    if allow_grad:
        emb = clip_model.encode_image(x)
    else:
        with torch.no_grad():
            emb = clip_model.encode_image(x)

    # CLIP on CUDA outputs float16 — cast to float32 to avoid dtype mismatch
    emb = emb.float()
    emb = emb / emb.norm(dim=-1, keepdim=True)
    emb = emb * (512 ** 0.5)  # rescaling
    return emb

## Step 7 – Image Preprocessing Transform

Define the transform pipeline for loading real images from the dataset:
1. Resize to 256×256 (matching StyleGAN2's output resolution)
2. Center crop to handle non-square images
3. Convert to tensor (`[0, 1]` range)
4. Normalize to `[-1, 1]` range (standard for StyleGAN2 and GAN training)

In [14]:
real_image_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(256),
    transforms.ToTensor(),  # gives [0,1]
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5]) # -> [-1,1]
])

## Step 8 – Define the Three Mapping Networks (`ThreeMappers`)

This is the core trainable component. We create **3 mapping networks**, each initialised as a **deep copy of the pretrained StyleGAN2 mapping network** (`G.mapping`).

### Key Change: Mappers Output a Delta (Mt(w)), Not a Full w+

Following the **StyleCLIP** formulation, each mapper now outputs a **manipulation step** `delta_w` rather than an absolute W+ latent.  
The final latent fed to the synthesis network is:

$$w_{final} = w_{avg} + M_t(w)$$

where `w_avg` is the mean latent from the pretrained StyleGAN2 model. This grounds all generations around the pretrained distribution and makes the L2 penalty on `Mt(w)` meaningful — large deltas are explicitly discouraged.

### Layer Assignments

```
Mapper 0 → w layers [0, 1, 2, 3]  → controls 4×4 and 8×8 blocks   (coarse structure)
Mapper 1 → w layers [4, 5, 6, 7]  → controls 16×16 and 32×32 blocks (medium features)
Mapper 2 → w layers [8–13]        → controls 64×64, 128×128, 256×256 (fine details)
```

The delta from each mapper is broadcast to all its assigned layers, forming the complete **W+** delta tensor of shape `(B, 14, 512)`.


In [15]:
class ThreeMappers(nn.Module):
    """
    3 mapping networks, each deep-copied from the pretrained StyleGAN2 mapping network.
    Each takes a 512-dim CLIP embedding and outputs a 512-dim *delta w* vector (Mt(w)).

    The final latent is:  w_final = w_avg + Mt(w)

    Layer assignments (for 256x256 resolution, num_ws=14):
        Mapper 0 → 4x4, 8x8           (w layers 0, 1, 2, 3)
        Mapper 1 → 16x16, 32x32       (w layers 4, 5, 6, 7)
        Mapper 2 → 64x64, 128x128, 256x256 (w layers 8–13)

    Truncation trick is applied to the raw mapper output before the delta is computed:
        raw_w  = mapper(clip_emb)
        trunc_w = w_avg + psi * (raw_w - w_avg)
        delta_w = trunc_w - w_avg          # ← this is Mt(w)
    """
    def __init__(self, pretrained_mapping, num_ws=14, w_dim=512, truncation_psi=0.7):
        super().__init__()
        self.num_ws = num_ws
        self.w_dim = w_dim
        self.truncation_psi = truncation_psi

        # Deep-copy pretrained mapping network 3 times
        self.mappers = nn.ModuleList([copy.deepcopy(pretrained_mapping) for _ in range(3)])

        # Enable gradient for all mapper parameters (they were frozen in the original G)
        for mapper in self.mappers:
            for p in mapper.parameters():
                p.requires_grad = True

        # Store w_avg from the pretrained model — used for truncation AND as the base latent
        self.register_buffer('w_avg', pretrained_mapping.w_avg.clone())  # (w_dim,)

        # Which synthesis layers each mapper feeds
        self.layer_assignments = [
            [0, 1, 2, 3],               # Mapper 0 → 4×4 + 8×8
            [4, 5, 6, 7],               # Mapper 1 → 16×16 + 32×32
            list(range(8, num_ws)),      # Mapper 2 → 64×64, 128×128, 256×256
        ]

    def forward(self, clip_embedding):
        """
        clip_embedding: (B, 512) float32 CLIP embedding
        Returns:
            delta_ws : (B, num_ws, w_dim) — the manipulation step Mt(w) in W+ space
        """
        delta_vectors = []
        for mapper in self.mappers:
            # Raw mapper output (no internal truncation — we apply it manually)
            w_broadcast = mapper(clip_embedding, c=None, truncation_psi=1.0)
            raw_w = w_broadcast[:, 0, :]  # (B, w_dim)

            # Truncation trick
            trunc_w = self.w_avg + self.truncation_psi * (raw_w - self.w_avg)

            # Delta = truncated output minus the mean latent  →  this is Mt(w)
            delta = trunc_w - self.w_avg  # (B, w_dim)
            delta_vectors.append(delta)

        # Build delta W+ tensor: (B, num_ws, w_dim)
        layer_to_delta = {}
        for mapper_idx, layers in enumerate(self.layer_assignments):
            for layer_idx in layers:
                layer_to_delta[layer_idx] = delta_vectors[mapper_idx]

        delta_ws = torch.stack([layer_to_delta[i] for i in range(self.num_ws)], dim=1)
        return delta_ws  # Mt(w)


## Step 9 – Instantiate the Three Mappers

Read `num_ws` (14 for 256×256), `w_dim` (512), and `c_dim` (0 for unconditional FFHQ) from the loaded StyleGAN2 model. Then create the `ThreeMappers` module:
- Deep-copies `G.mapping` three times
- Enables gradients on all mapper parameters (since G was frozen)
- Stores `w_avg` from the pretrained model for truncation
- Sets truncation factor $\psi = 0.7$

In [16]:
# Detect num_ws and w_dim from the pretrained model
num_ws = getattr(G.synthesis, 'num_ws', None) or getattr(G, 'num_ws', None) or 14
w_dim  = getattr(G, 'w_dim', 512)
c_dim  = getattr(G, 'c_dim', 0)

print(f'StyleGAN2 config: num_ws={num_ws}, w_dim={w_dim}, c_dim={c_dim}')

# Create 3 mapping networks, each initialised from the pretrained G.mapping
TRUNCATION_PSI = 0.7

mappers = ThreeMappers(
    pretrained_mapping=G.mapping,
    num_ws=num_ws,
    w_dim=w_dim,
    truncation_psi=TRUNCATION_PSI,
).to(device)

total_params     = sum(p.numel() for p in mappers.parameters())
trainable_params = sum(p.numel() for p in mappers.parameters() if p.requires_grad)
print(f'Three mapping networks created from pretrained StyleGAN2 mapping network.')
print(f'Total params: {total_params:,} | Trainable params: {trainable_params:,}')

StyleGAN2 config: num_ws=14, w_dim=512, c_dim=0
Three mapping networks created from pretrained StyleGAN2 mapping network.
Total params: 6,303,744 | Trainable params: 6,303,744


## Step 10 - Dataset Split and DataLoaders

Load the full face image dataset, then split it into **train/validation/test** subsets before training starts.

1. Build one dataset from `FFHQ_DIR`
2. Split into train, val, and test using `VAL_SPLIT` and `TEST_SPLIT`
3. Train loader is used for optimization
4. Validation and test loaders are reserved for evaluation and final testing

This ensures the model is trained only on the training subset and does not see validation/test images during optimization.

In [17]:
class SimpleImageDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        self.image_files = sorted([
            os.path.join(root, f)
            for f in os.listdir(root)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img = Image.open(self.image_files[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

full_dataset = SimpleImageDataset(FFHQ_DIR, transform=real_image_transform)
dataset_size = len(full_dataset)

if dataset_size < 3:
    raise ValueError(f"Need at least 3 images for train/val/test split, found {dataset_size}.")

val_size = max(1, int(dataset_size * VAL_SPLIT))
test_size = max(1, int(dataset_size * TEST_SPLIT))
train_size = dataset_size - val_size - test_size

if train_size < 1:
    raise ValueError(
        f"Invalid split sizes for dataset size {dataset_size}. "
        f"Got train={train_size}, val={val_size}, test={test_size}. "
        "Reduce VAL_SPLIT/TEST_SPLIT or use more images."
    )

split_generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size], generator=split_generator
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
 )
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
 )
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
 )

print(f"Dataset loaded: total={dataset_size} | train={len(train_dataset)} | val={len(val_dataset)} | test={len(test_dataset)}")

Dataset loaded: total=70000 | train=56000 | val=7000 | test=7000


## Step 11 – Training Loop

The training pipeline for each batch:

### Forward Pass
1. **Real images** → Frozen **CLIP ViT-B/32** → 512-dim embedding `clip_real` (detached, no grad)
2. `clip_real` → **3 Mapping Networks** (trainable) → **delta W+** tensor `delta_w = Mt(w)` of shape `(B, 14, 512)`
3. `w_plus = w_avg + delta_w` — add the mean latent to get the final manipulated latent
4. `w_plus` → Frozen **Synthesis Network** (`G.synthesis`) → Generated image `(B, 3, 256, 256)`

### Loss Computation
5. Generated image → Frozen **Discriminator** → Adversarial loss: $\mathcal{L}_{adv} = \text{softplus}(-D(x_{fake}))$
6. Generated image → Frozen **CLIP** → `clip_fake` → CLIP loss: $\mathcal{L}_{CLIP} = 1 - \cos(\text{clip\_fake},\; \text{clip\_real})$
7. L2 regularisation on the manipulation step: $\mathcal{L}_{L2} = \|M_t(w)\|_2^2$
8. **Total loss**: $\mathcal{L} = \mathcal{L}_{adv} + \mathcal{L}_{CLIP} + \lambda_{L2} \cdot \mathcal{L}_{L2}$

### Backward Pass
9. Gradients flow **only** through the 3 mapping networks (all other components are frozen)
10. Adam optimizer updates the mapper parameters

### Checkpointing
- Every epoch: save mapper weights and a sample image grid


In [18]:
optimizer = torch.optim.Adam(mappers.parameters(), lr=LR, betas=(0.9, 0.999))

step = 0
print_every = 50
save_every  = len(train_loader)

print(f'Training config: lr={LR}, lambda_l2={LAMBDA_L2}, truncation_psi={TRUNCATION_PSI}')
print(f'Frozen: G.synthesis, D, clip_model  |  Trainable: mappers (3 mapping networks)')


Training config: lr=0.002, Initial_lambda_clip=1, truncation_psi=0.7
Frozen: G.synthesis, D, clip_model  |  Trainable: mappers (3 mapping networks)


In [19]:
import wandb
import logging
from datetime import datetime
import os

RUN_NAME = f"clip_mapper_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
try:
    wandb.login()
except:
    pass

run = wandb.init(
    project="clip-stylegan-mapper",
    name=RUN_NAME,
    config={
        "lr": LR,
        "lambda_l2": LAMBDA_L2,
        "truncation_psi": TRUNCATION_PSI,
        "epochs": EPOCHS,
        "print_every": print_every,
        "save_every": save_every,
    }
)

LOG_FILE = os.path.join(SAVE_DIR, f"{BATCH_SIZE}-{LR}-{EPOCHS}-{LAMBDA_L2}-{RUN_NAME}.log")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE),
    ]
)

logger = logging.getLogger()


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/m2s/.netrc.
wandb: Currently logged in as: saffrotech (saffrotech-uom) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
from tqdm import tqdm

for epoch in range(1, EPOCHS + 1):
    logger.info(f"Starting Epoch Number {epoch}")
    pbar = tqdm(total=len(train_loader), desc=f"Epoch {epoch}/{EPOCHS}", leave=True)

    epoch_train_loss_total = 0.0
    epoch_train_adv_total  = 0.0
    epoch_train_clip_total = 0.0
    epoch_train_l2_total   = 0.0
    epoch_train_cos_total  = 0.0
    num_train_batches = 0

    for batch_idx, real_imgs in enumerate(train_loader):
        mappers.train()
        optimizer.zero_grad()

        real_imgs = real_imgs.to(device)  # (B, 3, 256, 256) in [-1, 1]

        # ---- 1) CLIP encode real images (no grad) ----
        clip_real = encode_with_clip_from_tensor(real_imgs, allow_grad=False).detach()

        # ---- 2) Mapper outputs manipulation step Mt(w) ----
        delta_w = mappers(clip_real)  # (B, num_ws, 512)

        # ---- 3) Build final latent: w_avg + Mt(w) ----
        # mappers.w_avg is (512,) → expand to (1, num_ws, 512) then broadcast over batch
        w_avg_expanded = mappers.w_avg.unsqueeze(0).unsqueeze(0).expand(real_imgs.shape[0], num_ws, -1)
        w_plus = w_avg_expanded + delta_w  # (B, num_ws, 512)

        # ---- 4) Synthesize images ----
        fake_images = G.synthesis(w_plus, noise_mode='const')

        # ---- 5) Adversarial loss ----
        try:
            d_fake = D(fake_images, c=torch.zeros(real_imgs.shape[0], c_dim, device=device))
        except TypeError:
            d_fake = D(fake_images)

        if isinstance(d_fake, (tuple, list)):
            d_fake = d_fake[0]

        g_adv_loss = F.softplus(-d_fake).mean()

        # ---- 6) CLIP loss: cosine distance G(w_avg + Mt(w)) vs real image ----
        # L_CLIP(w) = D_CLIP(G(w_avg + Mt(w)), x_real)
        clip_fake = encode_with_clip_from_tensor(fake_images, allow_grad=True)
        cos_sim   = F.cosine_similarity(clip_real, clip_fake, dim=-1)  # (B,)
        clip_loss = (1.0 - cos_sim).mean()                             # cosine distance

        # ---- 7) L2 regularisation on the manipulation step: lambda_L2 * ||Mt(w)||_2^2 ----
        l2_loss = (delta_w ** 2).sum(dim=-1).mean()  # sum over w_dim, mean over batch & layers

        # ---- 8) Total loss ----
        loss = g_adv_loss + clip_loss + LAMBDA_L2 * l2_loss

        # ---- 9) Backward ----
        loss.backward()
        optimizer.step()

        epoch_train_loss_total += loss.item()
        epoch_train_adv_total  += g_adv_loss.item()
        epoch_train_clip_total += clip_loss.item()
        epoch_train_l2_total   += l2_loss.item()
        epoch_train_cos_total  += cos_sim.mean().item()
        num_train_batches += 1

        # ---- Logging ----
        if step % print_every == 0:
            log_dict = {
                "loss_total":        loss.item(),
                "loss_adv":          g_adv_loss.item(),
                "loss_clip":         clip_loss.item(),
                "loss_l2":           l2_loss.item(),
                "cosine_similarity": cos_sim.mean().item(),
                "epoch":             epoch,
                "step":              step,
            }
            run.log(log_dict)
            logger.info(
                f"[Step {step}] Epoch {epoch} Batch {batch_idx} | "
                f"Loss: {loss.item():.4f} | Adv: {g_adv_loss.item():.4f} | "
                f"CLIP dist: {clip_loss.item():.4f} | L2: {l2_loss.item():.4f} | "
                f"Cos sim: {cos_sim.mean().item():.4f}"
            )

        # ---- Checkpointing ----
        if step % save_every == 0 and step > 0:
            ckpt_path = os.path.join(
                SAVE_DIR,
                f'step-{step}_l2-{LAMBDA_L2}_lr-{LR}_batchz-{BATCH_SIZE}.pt'
            )
            torch.save({
                'mappers_state_dict': mappers.state_dict(),
                'optimizer':          optimizer.state_dict(),
                'step':               step,
                'epoch':              epoch,
                'truncation_psi':     TRUNCATION_PSI,
            }, ckpt_path)
            print(f'  -> Saved checkpoint: {ckpt_path}')

            with torch.no_grad():
                sample_w_plus = w_plus[:min(4, w_plus.shape[0])]
                sample_imgs   = G.synthesis(sample_w_plus, noise_mode='const')
                grid_tensor   = utils.make_grid((sample_imgs + 1) / 2, nrow=sample_imgs.shape[0])
                grid_np       = (grid_tensor.cpu().numpy() * 255).astype('uint8').transpose(1, 2, 0)

                sample_path = os.path.join(SAVE_DIR, f'samples_step_{step}.png')
                utils.save_image(grid_tensor, sample_path)
                print(f'  -> Saved samples: {sample_path}')

            run.log({
                "sample_grid": wandb.Image(grid_np, caption=f"Step {step}"),
                "epoch":       epoch,
                "lambda_l2":   LAMBDA_L2,
            })

        step += 1
        pbar.update(1)
        pbar.set_postfix({
            "loss": loss.item(),
            "adv":  g_adv_loss.item(),
            "clip": clip_loss.item(),
            "l2":   l2_loss.item(),
            "cos":  cos_sim.mean().item(),
        })

    pbar.close()

    train_loss_epoch = epoch_train_loss_total / max(1, num_train_batches)
    train_adv_epoch  = epoch_train_adv_total  / max(1, num_train_batches)
    train_clip_epoch = epoch_train_clip_total / max(1, num_train_batches)
    train_l2_epoch   = epoch_train_l2_total   / max(1, num_train_batches)
    train_cos_epoch  = epoch_train_cos_total  / max(1, num_train_batches)

    # ---- Validation pass (no gradients) ----
    mappers.eval()
    val_loss_total = 0.0
    val_adv_total  = 0.0
    val_clip_total = 0.0
    val_l2_total   = 0.0
    val_cos_total  = 0.0
    num_val_batches = 0

    with torch.no_grad():
        for real_imgs in val_loader:
            real_imgs = real_imgs.to(device)

            clip_real = encode_with_clip_from_tensor(real_imgs, allow_grad=False).detach()
            delta_w   = mappers(clip_real)

            w_avg_expanded = mappers.w_avg.unsqueeze(0).unsqueeze(0).expand(real_imgs.shape[0], num_ws, -1)
            w_plus    = w_avg_expanded + delta_w

            fake_images = G.synthesis(w_plus, noise_mode='const')

            try:
                d_fake = D(fake_images, c=torch.zeros(real_imgs.shape[0], c_dim, device=device))
            except TypeError:
                d_fake = D(fake_images)
            if isinstance(d_fake, (tuple, list)):
                d_fake = d_fake[0]

            g_adv_loss = F.softplus(-d_fake).mean()
            clip_fake  = encode_with_clip_from_tensor(fake_images, allow_grad=False)
            cos_sim    = F.cosine_similarity(clip_real, clip_fake, dim=-1)
            clip_loss  = (1.0 - cos_sim).mean()
            l2_loss    = (delta_w ** 2).sum(dim=-1).mean()
            val_loss   = g_adv_loss + clip_loss + LAMBDA_L2 * l2_loss

            val_loss_total += val_loss.item()
            val_adv_total  += g_adv_loss.item()
            val_clip_total += clip_loss.item()
            val_l2_total   += l2_loss.item()
            val_cos_total  += cos_sim.mean().item()
            num_val_batches += 1

    val_loss_epoch = val_loss_total / max(1, num_val_batches)
    val_adv_epoch  = val_adv_total  / max(1, num_val_batches)
    val_clip_epoch = val_clip_total / max(1, num_val_batches)
    val_l2_epoch   = val_l2_total   / max(1, num_val_batches)
    val_cos_epoch  = val_cos_total  / max(1, num_val_batches)

    run.log({
        "train_loss_epoch":              train_loss_epoch,
        "train_adv_epoch":               train_adv_epoch,
        "train_clip_epoch":              train_clip_epoch,
        "train_l2_epoch":                train_l2_epoch,
        "train_cosine_similarity_epoch": train_cos_epoch,
        "val_loss_epoch":                val_loss_epoch,
        "val_adv_epoch":                 val_adv_epoch,
        "val_clip_epoch":                val_clip_epoch,
        "val_l2_epoch":                  val_l2_epoch,
        "val_cosine_similarity_epoch":   val_cos_epoch,
        "epoch":                         epoch,
        "step":                          step,
        "lambda_l2":                     LAMBDA_L2,
    })

    logger.info(
        f"[Epoch {epoch}] "
        f"TrainLoss: {train_loss_epoch:.4f} | ValLoss: {val_loss_epoch:.4f} | "
        f"TrainAdv: {train_adv_epoch:.4f} | ValAdv: {val_adv_epoch:.4f} | "
        f"TrainCLIP: {train_clip_epoch:.4f} | ValCLIP: {val_clip_epoch:.4f} | "
        f"TrainL2: {train_l2_epoch:.4f} | ValL2: {val_l2_epoch:.4f} | "
        f"TrainCos: {train_cos_epoch:.4f} | ValCos: {val_cos_epoch:.4f}"
    )

print("Training finished.")


Epoch 1/100:  99%|███████████████████████████████████████████████████████▎| 1729/1750 [12:41<00:09,  2.27it/s, loss=0.542, adv=0.136, clip=0.406, cos=0.594]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

Epoch 4/100:   0%|                                                          | 1/1750 [00:00<22:32,  1.29it/s, loss=0.343, adv=0.0179, clip=0.325, cos=0.675]

  → Saved checkpoint: /home/m2s/mind2sketch/Mapping-Neural-Networks/weights/split/3mapper_b_32/step-5250_lamda-1_lr-0.002_batchz-32.pt
  → Saved samples: /home/m2s/mind2sketch/Mapping-Neural-Networks/weights/split/3mapper_b_32/samples3_step_5250.png


Epoch 5/100:   0%|                                                         | 1/1750 [00:00<22:21,  1.30it/s, loss=0.275, adv=0.00845, clip=0.266, cos=0.734]

  → Saved checkpoint: /home/m2s/mind2sketch/Mapping-Neural-Networks/weights/split/3mapper_b_32/step-7000_lamda-1_lr-0.002_batchz-32.pt
  → Saved samples: /home/m2s/mind2sketch/Mapping-Neural-Networks/weights/split/3mapper_b_32/samples3_step_7000.png


Epoch 6/100:   0%|                                                         | 1/1750 [00:00<20:56,  1.39it/s, loss=0.231, adv=0.00905, clip=0.222, cos=0.778]

  → Saved checkpoint: /home/m2s/mind2sketch/Mapping-Neural-Networks/weights/split/3mapper_b_32/step-8750_lamda-1_lr-0.002_batchz-32.pt
  → Saved samples: /home/m2s/mind2sketch/Mapping-Neural-Networks/weights/split/3mapper_b_32/samples3_step_8750.png


Epoch 6/100:  12%|██████▎                                                | 202/1750 [01:29<11:05,  2.33it/s, loss=0.273, adv=0.00865, clip=0.265, cos=0.735]

In [ ]:
run.finish()